# Task 3: Forecast Future Market Trends

6–12 month TSLA forecast with confidence intervals.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import combine_close_prices, fetch_all_assets
from src.forecasting import fit_auto_arima, forecast_arima_with_intervals
from src.preprocessing import clean_asset_frame

FORECAST_DAYS = 252  # ~12 months of trading days

In [ ]:
asset_data = {t: clean_asset_frame(df) for t, df in fetch_all_assets().items()}
tsla = combine_close_prices(asset_data)['TSLA'].dropna()
train = tsla.loc[:'2024-12-31']

arima_model = fit_auto_arima(train, seasonal=True, m=5)
order = arima_model.order
seasonal_order = arima_model.seasonal_order
print('Selected order:', order, 'Seasonal:', seasonal_order)

forecast_mean, forecast_lower, forecast_upper = forecast_arima_with_intervals(
    train, steps=FORECAST_DAYS, order=order, seasonal_order=seasonal_order
)
future_index = pd.bdate_range(start=train.index[-1] + pd.Timedelta(days=1), periods=FORECAST_DAYS)
forecast_mean.index = future_index
forecast_lower.index = future_index
forecast_upper.index = future_index

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(tsla.index, tsla.values, label='Historical', color='black')
plt.plot(forecast_mean.index, forecast_mean.values, label='Forecast', color='royalblue')
plt.fill_between(forecast_mean.index, forecast_lower, forecast_upper, alpha=0.2, color='royalblue', label='95% CI')
plt.title('TSLA 12-Month Forecast with Confidence Intervals')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Trend Analysis

The forecast path reflects recent momentum in Tesla's price series. Confidence intervals widen as the horizon increases, indicating rising uncertainty for longer horizons — consistent with EMH limitations on price prediction.

**Opportunities:** Upward forecast slope may support a growth tilt for risk-tolerant clients.

**Risks:** Wide intervals imply substantial downside scenarios; TSLA volatility remains elevated.

**Reliability:** Near-term (1–3 month) forecasts are more reliable than 6–12 month projections.